In [1]:
import pandas as pd

# Labelled data from Day 4 (11 classes, Jan 2023 – May 2025)
df = pd.read_parquet("../data/interim/complaints_labeled.parquet")


In [2]:
print("Shape:", df.shape)                         # expect (855988, 5)
print("Unique labels:", df["label"].nunique())    # expect 11
print("Earliest:", df["Date received"].min().date())
print("Latest:  ", df["Date received"].max().date())


Shape: (855988, 5)
Unique labels: 11
Earliest: 2023-01-01
Latest:   2025-05-31


## Split decision: Option C

- **Train:** Jan 2023 – Dec 2024 (544,392 rows, 64%)
- **Validation:** Jan – Feb 2025 (142,407 rows, 17%)
- **Test:** Mar – May 2025 (169,189 rows, 20%)

Complaint volume grew 4–5×, so options A and B would make the test set over a third of the data and leave train at about half. Option C keeps the biggest training set, and every class still has at least 304 test rows. I put the Jan 2025 spike in validation rather than test, so the final score reflects the three most recent, normal months instead of one unusual event. The risk is tuning on a spike month, and the smallest test class may drop after de-duplication, so I'll re-check both.


In [4]:
date = df['Date received']

train = df[date < '2025-01-01'].copy()
val = df[(date>= '2025-01-01') & (date < '2025-03-01')].copy()
test = df[date >= '2025-03-01'].copy()


for name, part in [("train", train), ("val", val), ("test", test)]:
    print(name, len(part), part["Date received"].min().date(), part["Date received"].max().date())


assert len(train) + len(val) + len(test) == df.shape[0], "Rows lost or duplicated!"

train 544392 2023-01-01 2024-12-31
val 142407 2025-01-01 2025-02-28
test 169189 2025-03-01 2025-05-31


In [5]:
counts = pd.concat(
    [train['label'].value_counts(),val['label'].value_counts(),test['label'].value_counts()],
    axis=1,
    keys=['train','val','test']
)
print(counts)

                                                     train    val    test
label                                                                    
Credit reporting or other personal consumer rep...  417411  91690  135456
Debt collection                                      41625   8124   13507
Credit card                                          26867   3801    5049
Checking or savings account                          22513   9148    5115
Mortgage                                              9854   1260    1828
Money transfer, virtual currency, or money service    7230  24817    3067
Vehicle loan or lease                                 5777   1013    1349
Student loan                                          5755   1201    2144
Payday loan, title loan, personal loan, or adva...    3658    593     905
Prepaid card                                          2726    402     465
Debt or credit management                              976    358     304


In [7]:
shares = pd.concat(
    [p['label'].value_counts(normalize=True)*100 for p in [train, val, test]],
    axis=1,
    keys=['train','val','test']
).round(1)
print(shares)

                                                    train   val  test
label                                                                
Credit reporting or other personal consumer rep...   76.7  64.4  80.1
Debt collection                                       7.6   5.7   8.0
Credit card                                           4.9   2.7   3.0
Checking or savings account                           4.1   6.4   3.0
Mortgage                                              1.8   0.9   1.1
Money transfer, virtual currency, or money service    1.3  17.4   1.8
Vehicle loan or lease                                 1.1   0.7   0.8
Student loan                                          1.1   0.8   1.3
Payday loan, title loan, personal loan, or adva...    0.7   0.4   0.5
Prepaid card                                          0.5   0.3   0.3
Debt or credit management                             0.2   0.3   0.2


In [8]:
#DEDUPLICATION
before = len(train)
train = train.drop_duplicates(subset=["Consumer complaint narrative", "label"]).copy()
after = len(train)

print(f"Before: {before:,} | After: {after:,} | Removed: {before - after:,} ({(before - after) / before:.1%})")
print((train["label"].value_counts(normalize=True) * 100).round(1))

Before: 544,392 | After: 368,046 | Removed: 176,346 (32.4%)
label
Credit reporting or other personal consumer reports        68.7
Debt collection                                             9.1
Credit card                                                 6.4
Checking or savings account                                 6.1
Mortgage                                                    2.7
Money transfer, virtual currency, or money service          2.0
Vehicle loan or lease                                       1.6
Student loan                                                1.6
Payday loan, title loan, personal loan, or advance loan     1.0
Prepaid card                                                0.7
Debt or credit management                                   0.3
Name: proportion, dtype: float64


In [9]:
N = "Consumer complaint narrative"

# 1. Rebuild the ORIGINAL train (before de-duplication), where the votes are
train_full = df[df["Date received"] < "2025-01-01"].copy()

# 2. Count votes: for each row, how many times does its (text, label) pair appear?
train_full["n"] = train_full.groupby([N, "label"])["label"].transform("size")

# 3. Most votes first, then keep the first row per text = the winning label
train = train_full.sort_values("n", ascending=False).drop_duplicates(subset=N, keep="first").copy()

print("Rows:", len(train))
print("Duplicate narratives:", train[N].duplicated().sum())

# 4. Ties: texts where two labels have the same top vote count
pairs = train_full.groupby([N, "label"]).size().reset_index(name="n")
top = pairs[pairs["n"] == pairs.groupby(N)["n"].transform("max")]
print("Ties:", top.groupby(N).size().gt(1).sum())


Rows: 367414
Duplicate narratives: 0
Ties: 117


In [10]:
N = "Consumer complaint narrative"

# 1. Test rows whose text ALSO appears in train ("seen before")
seen = test[N].isin(train[N])
print(f"Seen in train: {seen.sum():,} rows ({seen.mean():.1%} of test)")

# 2. Remove them  3. Remove plain copies (same text + same label), NO label fixing
test_new = test[~seen].drop_duplicates(subset=[N, "label"]).copy()

# 4. Size and smallest class
print("Final rows:", len(test_new))
print(test_new["label"].value_counts().tail(3))


Seen in train: 30,249 rows (17.9% of test)
Final rows: 87091
label
Payday loan, title loan, personal loan, or advance loan    905
Prepaid card                                               465
Debt or credit management                                  304
Name: count, dtype: int64


In [11]:
from pathlib import Path
Path("../data/processed").mkdir(exist_ok=True)

train = train.drop(columns="n")   # remove the vote-count helper column

for name, part in [("train", train), ("val", val), ("test", test), ("test_dedup", test_new)]:
    part.to_parquet(f"../data/processed/{name}.parquet", index=False)
    print(name, pd.read_parquet(f"../data/processed/{name}.parquet").shape)


train (367414, 5)
val (142407, 5)
test (169189, 5)
test_dedup (87091, 5)
